# CHG — Análisis Raster con Python
## Día 3 · Módulo B: Rasterio — Landsat sobre Doñana

**Curso de Python para el Análisis Espacial**
Confederación Hidrográfica del Guadalquivir

---

En este notebook usaremos una **imagen Landsat 9 OLI del 27 de marzo de 2025 recortada sobre la Reserva de la Biosfera de Doñana**, ya reescalada a reflectancia (valores 0–1, `float32`), con 6 bandas espectrales **+ una 7ª banda de calidad (QA_PIXEL)**:

| Índice (Python) | Banda     | λ (µm)      | Para qué |
|-----------------|-----------|-------------|----------|
| `1`             | Blue  (B2)| 0.45–0.51   | Composiciones color natural |
| `2`             | Green (B3)| 0.53–0.59   | NDWI, color natural |
| `3`             | Red   (B4)| 0.64–0.67   | NDVI, color natural |
| `4`             | NIR   (B5)| 0.85–0.88   | NDVI, NDWI, biomasa |
| `5`             | SWIR1 (B6)| 1.57–1.65   | MNDWI, humedad |
| `6`             | SWIR2 (B7)| 2.11–2.29   | Falsos color, geología |
| `7`             | QA_PIXEL  | —           | **Máscara de calidad** (Fmask): nubes, sombras, cirrus |

> Rasterio numera las bandas **empezando por 1** (no por 0), porque internamente usa GDAL.
>
> 📅 **¿Por qué esta fecha?** Marzo 2025 con la marisma muy inundada por las lluvias del invierno — ideal para detectar agua. Además tiene algo de nubosidad, lo que nos permite practicar el **filtrado por máscara de calidad**.

## Contenidos

1. Setup en Colab (descarga desde Nextcloud) o local
2. **GDAL bajo el capó** — `gdalinfo`, `gdal_translate`, `gdalwarp`
3. Abrir el raster: `crs`, `transform`, `bounds`, `profile`
4. Leer bandas — una a una y *stack* completo
5. Visualización banda a banda y composiciones RGB
6. **Filtrado de nubes y sombras** — decodificar bits de QA_PIXEL
7. Álgebra de bandas: **NDVI**, **NDWI**, **MNDWI**, **CIgreen**
8. Guardar un raster derivado a disco
9. **Máscara de agua** → **poligonización** con `rasterio.features.shapes()`
10. Recorte de raster por un vectorial (`rasterio.mask.mask`)
11. **Estadísticas zonales** con `rasterstats`
12. Ejercicio final integrador
13. Apéndice: cómo se preparó el recorte Landsat


---
## 1. Setup — datos en Colab o en local

Este notebook está pensado para ejecutarse **tanto en Colab como en local**. La celda siguiente detecta el entorno:
- Si estás en **Colab**, descarga la imagen Landsat y los vectoriales desde Nextcloud.
- Si estás en **local**, asume que los vectoriales viven en `../dia_2_analisis_vectorial/data/` y la Landsat en `./data/`.

> ✏️ **Profesor: sustituir las URL `NEXTCLOUD_…` por los enlaces compartidos reales antes de la clase.**


In [ ]:
import os, sys, subprocess, pathlib

# --- URLs Nextcloud (rellenar con los enlaces públicos /download de cada archivo) ---
NEXTCLOUD_LANDSAT      = "https://nextcloud.tu-dominio.es/s/REEMPLAZAR_LANDSAT/download"
NEXTCLOUD_MUNICIPIOS   = "https://nextcloud.tu-dominio.es/s/REEMPLAZAR_MUNICIPIOS/download"
NEXTCLOUD_CUENCAS      = "https://nextcloud.tu-dominio.es/s/REEMPLAZAR_CUENCAS/download"
NEXTCLOUD_EMBALSES     = "https://nextcloud.tu-dominio.es/s/REEMPLAZAR_EMBALSES/download"
NEXTCLOUD_RBIOS_ZIP    = "https://nextcloud.tu-dominio.es/s/REEMPLAZAR_RBIOS_ZIP/download"  # rbios.shp + .shx + .dbf + .prj (zipped)

try:
    import google.colab  # noqa
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    # Instalación de dependencias geoespaciales no incluidas por defecto en Colab
    subprocess.run(["pip", "-q", "install", "rasterio", "rasterstats", "geopandas", "matplotlib"], check=False)

    def _fetch(url, dst):
        if not os.path.exists(dst):
            print(f"⬇️  Descargando {dst}…")
            subprocess.run(["wget", "-q", "-O", dst, url], check=True)
        else:
            print(f"✔  {dst} ya está descargado")

    _fetch(NEXTCLOUD_LANDSAT,    "landsat_donana_marisma.tif")
    _fetch(NEXTCLOUD_MUNICIPIOS, "terminos_municipales_andalucia.gpkg")
    _fetch(NEXTCLOUD_CUENCAS,    "cuencas_guadalquivir.gpkg")
    _fetch(NEXTCLOUD_EMBALSES,   "embalses_guadalquivir.gpkg")
    # El shapefile rbios se descarga como ZIP (4 ficheros) y se descomprime
    if not os.path.exists("rbios.shp"):
        _fetch(NEXTCLOUD_RBIOS_ZIP, "rbios.zip")
        subprocess.run(["unzip", "-o", "rbios.zip"], check=True)

    LANDSAT_PATH  = "landsat_donana_marisma.tif"
    MUNICIPIOS    = "terminos_municipales_andalucia.gpkg"
    CUENCAS       = "cuencas_guadalquivir.gpkg"
    EMBALSES      = "embalses_guadalquivir.gpkg"
    RBIOS         = "rbios.shp"
else:
    DATA_VECT = pathlib.Path("../dia_2_analisis_vectorial/data")
    DATA_RAST = pathlib.Path("./data")
    LANDSAT_PATH  = str(DATA_RAST / "landsat_donana_marisma.tif")
    MUNICIPIOS    = str(DATA_VECT / "terminos_municipales_andalucia.gpkg")
    CUENCAS       = str(DATA_VECT / "cuencas_guadalquivir.gpkg")
    EMBALSES      = str(DATA_VECT / "embalses_guadalquivir.gpkg")
    RBIOS         = str(DATA_VECT / "rbios.shp")

print("Entorno:", "Colab" if EN_COLAB else "Local")
print("Raster: ", LANDSAT_PATH, "→ existe:", os.path.exists(LANDSAT_PATH))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import geopandas as gpd

print(f'rasterio {rasterio.__version__}')
print(f'geopandas {gpd.__version__}')


---
## 2. GDAL — el motor que hay debajo

Antes de tirar de `rasterio`, una breve pausa para ver qué hay realmente bajo el capó.

**GDAL** (Geospatial Data Abstraction Library) es una librería C/C++ de los años 90 que sabe leer, escribir y transformar **más de 160 formatos raster** (y otros tantos vectoriales). Es la base sobre la que se apoyan casi todas las herramientas SIG modernas:

```
            GDAL  (C/C++, formato + álgebra + reproyección)
              │
   ┌──────────┼──────────────────────────────────────┐
   │          │                                      │
  QGIS    rasterio (Python)                  GRASS, SAGA, ArcGIS,
          rioxarray, fiona, gdalwarp,        Google Earth Engine,
          osgeo.gdal …                       PostGIS raster, ...
```

GDAL viene **preinstalado en Colab** (paquete `gdal-bin`). Podemos llamarlo desde el notebook con el prefijo `!`.


In [ ]:
!gdalinfo --version

### `gdalinfo` — el "describir" del raster

Equivale a abrirlo con rasterio y consultar `.crs`, `.bounds`, `.transform`, estadísticas… todo de un golpe.


In [ ]:
!gdalinfo {LANDSAT_PATH}

> Lo que ves aquí es exactamente lo mismo que rasterio expone en `.profile`, `.bounds`, `.transform`, etc. La diferencia: rasterio te lo da como objetos Python manejables.


### `gdal_translate` — convertir / recortar / extraer bandas

Algunos ejemplos clásicos (sustituye `!` por terminal fuera de Jupyter):


In [ ]:
# Extraer SOLO la banda 4 (NIR) a un fichero nuevo, comprimido
!gdal_translate -b 4 -co COMPRESS=DEFLATE -co TILED=YES \
    {LANDSAT_PATH} solo_nir.tif

# Pedimos a GDAL que nos informe del resultado
!gdalinfo solo_nir.tif | head -15


### `gdalwarp` — reproyectar y/o recortar

Reproyectar a coordenadas geográficas (WGS84) — útil para visualizar en folium/leaflet sin pasar por Python:

```bash
gdalwarp -t_srs EPSG:4326 -r bilinear  landsat_donana_marisma.tif  landsat_wgs84.tif
```

Recortar por un vectorial (lo mismo que `rasterio.mask.mask`, pero en CLI):

```bash
gdalwarp -cutline almonte.gpkg -crop_to_cutline  landsat_donana_marisma.tif  landsat_almonte.tif
```

### Cuándo GDAL CLI y cuándo rasterio

| Tarea | Mejor con |
|-------|-----------|
| Inspección rápida de un fichero | `gdalinfo` |
| Conversión de formato puntual | `gdal_translate` |
| Reproyectar/recortar en *batch* (scripts shell, automatización en CI) | `gdalwarp` |
| Cualquier análisis donde mezclas datos con código Python | `rasterio` |
| Acceder al array NumPy del raster para hacer álgebra | `rasterio` |
| Integrarlo con pandas/geopandas en un workflow | `rasterio` |

> Resumen: **rasterio = GDAL con cara de Python**. Saber que GDAL existe te abre la puerta a leer documentación, copiar comandos de Stack Overflow y entender cualquier librería raster del ecosistema.


---
## 3. Abrir el raster y leer sus metadatos

`rasterio.open()` no carga los píxeles en memoria — solo los metadatos. Esto te permite consultar el CRS, la resolución y los bounds **sin coste**, antes de decidir cuánto leer.


In [ ]:
src = rasterio.open(LANDSAT_PATH)
print('CRS:       ', src.crs)
print('Bounds:    ', src.bounds)
print('Width x H: ', src.width, 'x', src.height)
print('Bandas:    ', src.count)
print('Dtype:     ', src.dtypes[0])
print('NoData:    ', src.nodatavals)
print('Transform: ', src.transform)


### El `Affine` transform

```
Affine(a, b, c,
       d, e, f)
```

- `a` = tamaño de píxel en X (resolución horizontal, m si el CRS es proyectado)
- `e` = tamaño de píxel en Y (negativo, porque los rasters se almacenan de arriba abajo)
- `c`, `f` = coordenadas del píxel superior izquierdo

Con esto, rasterio convierte fila/columna a coordenadas del mundo (y viceversa).


In [ ]:
# Coordenadas del píxel central
fila_c, col_c = src.height // 2, src.width // 2
x, y = src.transform * (col_c, fila_c)
print(f'Píxel central (col={col_c}, fila={fila_c}) → ({x:.1f}, {y:.1f}) en {src.crs}')

# Y al revés
fila, col = src.index(x, y)
print(f'Coords ({x:.1f}, {y:.1f}) → píxel (col={col}, fila={fila})')


In [ ]:
# El profile: el "carnet de identidad" del raster. Lo reutilizaremos al escribir derivados.
profile = src.profile
print(profile)


---
## 4. Leer bandas

Tres formas:
1. `src.read(n)` → numpy 2D de la banda `n`.
2. `src.read([1,2,3])` → numpy 3D con varias bandas en orden.
3. `src.read()` → numpy 3D con **todas** las bandas. Forma `(bandas, alto, ancho)`.


In [ ]:
red   = src.read(3)
nir   = src.read(4)
green = src.read(2)
swir1 = src.read(5)

print('NIR shape:', nir.shape, '| dtype:', nir.dtype)
print(f'NIR  rango: {nir.min():.3f} – {nir.max():.3f}')
print(f'Red  rango: {red.min():.3f} – {red.max():.3f}')


In [ ]:
# Stack completo de las 7 bandas en una sola llamada (6 espectrales + QA_PIXEL)
stack = src.read()       # shape (7, H, W)
print('Stack shape:', stack.shape)
print('Memoria:    {:.1f} MB'.format(stack.nbytes / 1e6))


---
## 5. Visualización

### 5.1 Una banda con su colormap


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
img = ax.imshow(nir, cmap='gray', vmin=0, vmax=0.5)
plt.colorbar(img, ax=ax, fraction=0.04, label='Reflectancia NIR')
ax.set_title('Banda NIR (B5) — Landsat sobre Doñana')
ax.set_axis_off()
plt.show()


### 5.2 Las 6 bandas a la vez


In [ ]:
nombres = ['Blue (B2)', 'Green (B3)', 'Red (B4)', 'NIR (B5)', 'SWIR1 (B6)', 'SWIR2 (B7)']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for i, ax in enumerate(axes.flat):
    banda = stack[i]
    # vmax por percentil para evitar que un píxel saturado nos coma el contraste
    vmax = np.percentile(banda, 98)
    ax.imshow(banda, cmap='gray', vmin=0, vmax=vmax)
    ax.set_title(nombres[i])
    ax.set_axis_off()
plt.tight_layout()
plt.show()


### 5.3 Composición color natural (RGB)

Para que matplotlib muestre una RGB necesita:
- forma `(alto, ancho, 3)` → usamos `np.dstack`
- valores en `[0, 1]` → recortamos y/o estiramos contraste


In [ ]:
def estirar(banda, p_min=2, p_max=98):
    """Estiramiento lineal por percentiles para realzar el contraste visual."""
    vmin, vmax = np.percentile(banda, [p_min, p_max])
    return np.clip((banda - vmin) / (vmax - vmin), 0, 1)

rgb_natural = np.dstack([
    estirar(red),    # R
    estirar(green),  # G
    estirar(src.read(1)),  # B (blue)
])

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(rgb_natural)
ax.set_title('Color natural (R=B4, G=B3, B=B2) — Doñana')
ax.set_axis_off()
plt.show()


### 5.4 Falso color infrarrojo — la vegetación en rojo

Sustituir el rojo visible por el NIR resalta toda la vegetación viva en tonos rojos. Es el "clásico" de la teledetección.


In [ ]:
rgb_falso = np.dstack([
    estirar(nir),     # R ← NIR
    estirar(red),     # G ← Red
    estirar(green),   # B ← Green
])

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(rgb_falso)
ax.set_title('Falso color IR (R=NIR, G=Red, B=Green) — vegetación en rojo')
ax.set_axis_off()
plt.show()


---
## 6. Filtrado de nubes y sombras — la banda QA_PIXEL

Antes de calcular índices conviene **quitar los píxeles malos** (nubes, sombras, cirrus). Si no los filtramos, los meten en las medias zonales, en los histogramas y en las máscaras de agua → resultados sucios.

La 7ª banda de nuestro raster es la **QA_PIXEL** (también llamada "Fmask" en la comunidad), una banda de calidad de Landsat Collection 2 que **codifica varias condiciones de calidad en los bits de un entero de 16 bits**. Es decir: el mismo número de píxel contiene a la vez información sobre nube, sombra, cirrus, agua…

### 6.1 Estructura de QA_PIXEL en Landsat 8/9 Collection 2

| Bit  | Significado            | Cómo se lee |
|------|------------------------|-------------|
| 0    | Fill (sin datos)       | `(qa >> 0) & 1` |
| 1    | Dilated Cloud          | `(qa >> 1) & 1` |
| 2    | Cirrus                 | `(qa >> 2) & 1` |
| 3    | **Cloud**              | `(qa >> 3) & 1` |
| 4    | **Cloud Shadow**       | `(qa >> 4) & 1` |
| 5    | Snow                   | `(qa >> 5) & 1` |
| 6    | Clear (terreno limpio) | `(qa >> 6) & 1` |
| 7    | Water                  | `(qa >> 7) & 1` |
| 8-9  | Cloud confidence       | `(qa >> 8) & 0b11` |
| 10-11| Shadow confidence      | `(qa >> 10) & 0b11` |
| 12-13| Snow confidence        | `(qa >> 12) & 0b11` |
| 14-15| Cirrus confidence      | `(qa >> 14) & 0b11` |

> `>>` es **desplazamiento de bits a la derecha**: corre el número N bits hacia la derecha y `& 1` se queda con el bit más bajo. Es la forma canónica de "extraer una flag" empaquetada en un entero.


In [ ]:
# Leemos la QA. Está guardada como float32 (mismo dtype que el resto), pero los
# valores válidos son enteros. La convertimos a int32 para hacer operaciones bit a bit.
qa = src.read(7)
mask_valido = ~np.isnan(qa)            # píxeles dentro de la escena
qa_int = np.zeros_like(qa, dtype='int32')
qa_int[mask_valido] = qa[mask_valido].astype('int32')

# Valores más frecuentes
vals, counts = np.unique(qa_int[mask_valido], return_counts=True)
top = sorted(zip(counts, vals), reverse=True)[:8]
print('Valores QA más frecuentes en la escena:')
for c, v in top:
    print(f'  {v:>6} (0b{v:016b}) — {c:>10,} px ({100*c/mask_valido.sum():.2f}%)')


### 6.2 Decodificar los bits — extraer cada flag

Construimos una flag de "nube" haciendo álgebra bit a bit:


In [ ]:
# Extraemos cada flag como un raster booleano del mismo tamaño que la imagen
is_cloud         = ((qa_int >> 3) & 1).astype(bool)
is_cloud_shadow  = ((qa_int >> 4) & 1).astype(bool)
is_cirrus        = ((qa_int >> 2) & 1).astype(bool)
is_dilated_cloud = ((qa_int >> 1) & 1).astype(bool)
is_snow          = ((qa_int >> 5) & 1).astype(bool)
is_water         = ((qa_int >> 7) & 1).astype(bool)
is_clear         = ((qa_int >> 6) & 1).astype(bool)

total_valid = mask_valido.sum()
for nombre, flag in [('cloud', is_cloud), ('cloud_shadow', is_cloud_shadow),
                     ('cirrus', is_cirrus), ('dilated_cloud', is_dilated_cloud),
                     ('snow', is_snow), ('water', is_water), ('clear', is_clear)]:
    n = flag.sum()
    print(f'  {nombre:15s}: {n:>10,} px ({100*n/total_valid:5.2f}%)')


### 6.3 Visualizar las nubes y las sombras


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

axes[0].imshow(rgb_natural)
axes[0].set_title('Color natural (referencia)')
axes[0].set_axis_off()

axes[1].imshow(is_cloud, cmap='gray_r')
axes[1].set_title(f'Nubes  ({is_cloud.sum():,} px)')
axes[1].set_axis_off()

axes[2].imshow(is_cloud_shadow, cmap='gray_r')
axes[2].set_title(f'Sombras de nube  ({is_cloud_shadow.sum():,} px)')
axes[2].set_axis_off()

plt.tight_layout()
plt.show()


### 6.4 Construir la máscara de "píxel bueno"

Combinamos las flags para quedarnos solo con píxeles **dentro de la escena** y **sin nube, sin sombra, sin cirrus, sin nieve**:


In [ ]:
pixel_bueno = (
    mask_valido
    & ~is_cloud
    & ~is_cloud_shadow
    & ~is_cirrus
    & ~is_dilated_cloud
    & ~is_snow
)

n_bueno = pixel_bueno.sum()
print(f'Píxeles válidos:  {mask_valido.sum():>10,}')
print(f'Píxeles "buenos": {n_bueno:>10,} ({100*n_bueno/mask_valido.sum():.1f}%)')

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(pixel_bueno, cmap='gray')
ax.set_title(f'Máscara de píxel bueno ({100*n_bueno/mask_valido.sum():.0f}% de la escena)')
ax.set_axis_off()
plt.show()


### 6.5 Aplicar la máscara — el NDVI antes y después

Vamos a calcular un NDVI rápido y comparar **sin filtrar** vs **filtrado** por la máscara de calidad. Donde no haya píxel bueno, ponemos `np.nan`:


In [ ]:
with np.errstate(divide='ignore', invalid='ignore'):
    ndvi_sucio   = (nir - red) / (nir + red)
    ndvi_limpio  = np.where(pixel_bueno, ndvi_sucio, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

im0 = axes[0].imshow(ndvi_sucio,  cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0].set_title('NDVI sin filtrar (con nubes y sombras)')
axes[0].set_axis_off()

im1 = axes[1].imshow(ndvi_limpio, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[1].set_title('NDVI filtrado por QA_PIXEL')
axes[1].set_axis_off()

plt.colorbar(im1, ax=axes, fraction=0.02, label='NDVI')
plt.show()

# Comparación numérica
print(f'NDVI sucio  — media: {np.nanmean(ndvi_sucio):.3f}')
print(f'NDVI limpio — media: {np.nanmean(ndvi_limpio):.3f}')
print(f'Diferencia debida a nubes/sombras: {np.nanmean(ndvi_sucio) - np.nanmean(ndvi_limpio):.3f}')


> 🎯 **Por qué importa:** las nubes saturan en rojo y verde → NDVI ~0; las sombras tienen reflectancia muy baja en todas las bandas → NDVI ruidoso. Si calculas medias zonales sobre una escena sin filtrar, las cuencas con nubosidad **parecen tener menos vegetación de la que tienen**. El filtrado QA es **el primer paso de cualquier flujo serio de teledetección**.
>
> 💡 **Versión "una sola línea"** — si solo necesitas un filtro rápido y has visto que la mayoría de tu escena tiene `qa == 21824` (clear land) o `qa == 21952` (clear water):
>
> ```python
> pixel_bueno = (qa_int == 21824) | (qa_int == 21952)
> ```
>
> Es menos elegante pero funciona para escenas concretas. La versión bit a bit es la que escala a cualquier escena.


---
## 7. Álgebra de bandas — los índices clásicos

A partir de aquí calcularemos los índices **sobre los datos sin filtrar**, para que los pasos sean claros. Después aplicaremos `pixel_bueno` cuando queramos resultados limpios. En la práctica, lo habitual es **filtrar antes** del cálculo o **enmascarar después** — ambas formas son equivalentes.

### 7.1 NDVI — vigor vegetal

$$NDVI = \dfrac{NIR - Red}{NIR + Red}$$

Rango teórico [-1, 1]. Vegetación sana ~0.6–0.9, suelo ~0.1–0.3, agua ~0 o negativo.

> Antes de dividir, **siempre** convertir a `float`. Si las bandas vienen como `uint16` la división trunca a entero.


In [ ]:
# np.errstate suprime el warning de "división por cero" en píxeles nodata
with np.errstate(divide='ignore', invalid='ignore'):
    ndvi = (nir - red) / (nir + red)

print(f'NDVI rango: {np.nanmin(ndvi):.2f} – {np.nanmax(ndvi):.2f}')
print(f'NDVI media: {np.nanmean(ndvi):.2f}')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
img = ax.imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
plt.colorbar(img, ax=ax, fraction=0.04, label='NDVI')
ax.set_title('NDVI — Doñana')
ax.set_axis_off()
plt.show()


### 7.2 NDWI de McFeeters (1996) — detecta agua

$$NDWI = \dfrac{Green - NIR}{Green + NIR}$$

Agua → positivo. Vegetación y suelo → negativo. Es el clásico para masas de agua claras.


In [ ]:
with np.errstate(divide='ignore', invalid='ignore'):
    ndwi = (green - nir) / (green + nir)

fig, ax = plt.subplots(figsize=(9, 9))
img = ax.imshow(ndwi, cmap='Blues', vmin=-0.3, vmax=0.6)
plt.colorbar(img, ax=ax, fraction=0.04, label='NDWI')
ax.set_title('NDWI (McFeeters) — Doñana')
ax.set_axis_off()
plt.show()


### 7.3 MNDWI de Xu (2006) — agua mejor en zonas urbanas/turbias

$$MNDWI = \dfrac{Green - SWIR_1}{Green + SWIR_1}$$

El SWIR es **muy** absorbido por el agua, así que el contraste agua/no-agua es mayor que en NDWI. Es el índice preferido para marismas, aguas someras y cuerpos urbanos.


In [ ]:
with np.errstate(divide='ignore', invalid='ignore'):
    mndwi = (green - swir1) / (green + swir1)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(ndwi, cmap='Blues', vmin=-0.3, vmax=0.6)
axes[0].set_title('NDWI (McFeeters)')
axes[0].set_axis_off()

im = axes[1].imshow(mndwi, cmap='Blues', vmin=-0.3, vmax=0.8)
axes[1].set_title('MNDWI (Xu) — mejor para marismas')
axes[1].set_axis_off()
plt.colorbar(im, ax=axes[1], fraction=0.04)
plt.tight_layout()
plt.show()


### 7.4 CIgreen — proxy de clorofila

$$CI_{green} = \dfrac{NIR}{Green} - 1$$

Muy correlacionado con el contenido de clorofila en hoja. Útil para seguimiento de cultivos y, también, para detectar **blooms de fitoplancton/algas** en láminas de agua.


In [ ]:
with np.errstate(divide='ignore', invalid='ignore'):
    cigreen = nir / green - 1

fig, ax = plt.subplots(figsize=(9, 9))
img = ax.imshow(cigreen, cmap='YlGn', vmin=0, vmax=8)
plt.colorbar(img, ax=ax, fraction=0.04, label='CIgreen')
ax.set_title('Chlorophyll Index green — Doñana')
ax.set_axis_off()
plt.show()


---
## 8. Guardar un raster derivado

Para escribir un GeoTIFF nuevo necesitamos un **`profile`** (los metadatos que describen el archivo). Lo más cómodo es **partir del profile original** y modificar lo que cambie: el número de bandas y el `dtype`.


In [ ]:
profile_ndvi = src.profile.copy()
profile_ndvi.update(
    count=1,            # 1 sola banda
    dtype='float32',    # NDVI es float
    nodata=np.nan,      # marcamos NoData como NaN
    compress='deflate',
)

with rasterio.open('ndvi_donana.tif', 'w', **profile_ndvi) as dst:
    dst.write(ndvi.astype('float32'), 1)
    dst.set_band_description(1, 'NDVI')

import os
print(f'Guardado: ndvi_donana.tif ({os.path.getsize("ndvi_donana.tif")/1e6:.1f} MB)')


---
## 9. Máscara de agua → polígonos

Vamos a:
1. Crear una **máscara binaria** de agua con un umbral sobre el MNDWI.
2. **Poligonizarla** con `rasterio.features.shapes()` para obtener un GeoDataFrame de masas de agua.

Esto es lo que QGIS llama "Raster → Vectorial (Poligonizar)" — pero en Python, en una celda.


In [ ]:
# Umbral típico: MNDWI > 0 → agua. Subimos un poco para ser más conservadores.
UMBRAL = 0.1
mask_agua = (mndwi > UMBRAL).astype('uint8')   # uint8 — necesario para shapes()

print(f'Píxeles de agua: {mask_agua.sum():,} de {mask_agua.size:,} '
      f'({100*mask_agua.sum()/mask_agua.size:.1f}%)')

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(mask_agua, cmap='Blues', vmin=0, vmax=1)
ax.set_title(f'Máscara binaria de agua (MNDWI > {UMBRAL})')
ax.set_axis_off()
plt.show()


### Poligonización con `rasterio.features.shapes`

Genera un iterador `(geometría, valor)` por cada **región conectada** del raster. Filtramos quedándonos solo con `valor == 1` (agua).


In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape

geoms_agua = []
for geom_dict, val in shapes(mask_agua, mask=(mask_agua == 1), transform=src.transform):
    geoms_agua.append({'geometry': shape(geom_dict), 'mndwi_mean': None})

gdf_agua = gpd.GeoDataFrame(geoms_agua, crs=src.crs)
print(f'{len(gdf_agua)} polígonos de agua detectados')

# Filtrar polígonos pequeños (ruido, píxeles aislados)
MIN_AREA_M2 = 30 * 30 * 10   # al menos 10 píxeles Landsat (≈ 0.9 ha)
gdf_agua = gdf_agua[gdf_agua.area > MIN_AREA_M2].reset_index(drop=True)
gdf_agua['area_ha'] = gdf_agua.area / 1e4

print(f'Tras filtrar < {MIN_AREA_M2/1e4:.1f} ha: {len(gdf_agua)} polígonos')
gdf_agua.sort_values('area_ha', ascending=False).head(10)


In [ ]:
# Visualización: polígonos de agua sobre el MNDWI
fig, ax = plt.subplots(figsize=(9, 9))
show(mndwi, transform=src.transform, ax=ax, cmap='Blues', vmin=-0.2, vmax=0.6)
gdf_agua.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=0.7)
ax.set_title(f'Cuerpos de agua detectados ({len(gdf_agua)} polígonos)')
plt.show()


In [ ]:
# Guardar como GeoPackage
gdf_agua.to_file('cuerpos_agua_donana.gpkg', driver='GPKG')
print('Guardado: cuerpos_agua_donana.gpkg')


---
## 10. Recortar el raster por un vectorial

`rasterio.mask.mask(src, geometrias, crop=True)` recorta el raster a la extensión de las geometrías y pone a NoData fuera de ellas.

Aquí recortamos por el **término municipal de Almonte** (Doñana cae mayormente sobre Almonte).


In [ ]:
from rasterio.mask import mask as rio_mask

municipios = gpd.read_file(MUNICIPIOS)
print('CRS municipios:', municipios.crs, '|  CRS raster:', src.crs)

# Reproyectamos los municipios al CRS del raster
municipios = municipios.to_crs(src.crs)

# Filtramos Almonte
almonte = municipios[municipios['nombre'] == 'Almonte']
print(f'Almonte: {len(almonte)} entidad(es), área {almonte.area.iloc[0]/1e6:.1f} km²')


In [ ]:
geom_almonte = [almonte.geometry.iloc[0].__geo_interface__]

with rasterio.open(LANDSAT_PATH) as s:
    img_recortada, t_recortada = rio_mask(s, geom_almonte, crop=True, nodata=np.nan, filled=True)
    prof_rec = s.profile.copy()

prof_rec.update({
    'height':    img_recortada.shape[1],
    'width':     img_recortada.shape[2],
    'transform': t_recortada,
    'dtype':     'float32',
    'nodata':    np.nan,
    'compress':  'deflate',
})

with rasterio.open('landsat_almonte.tif', 'w', **prof_rec) as dst:
    dst.write(img_recortada.astype('float32'))

print('shape recortada:', img_recortada.shape)
print('Guardado: landsat_almonte.tif')


In [ ]:
# NDVI sobre el recorte de Almonte
red_a = img_recortada[2]
nir_a = img_recortada[3]
with np.errstate(divide='ignore', invalid='ignore'):
    ndvi_almonte = (nir_a - red_a) / (nir_a + red_a)

fig, ax = plt.subplots(figsize=(9, 9))
show(ndvi_almonte, transform=t_recortada, ax=ax, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
almonte.boundary.plot(ax=ax, color='black', linewidth=1.2)
ax.set_title('NDVI — Término municipal de Almonte')
plt.show()


---
## 11. Estadísticas zonales — NDVI medio por municipio

`rasterstats` calcula estadísticas de un raster dentro de cada polígono de un vectorial. Ideal para informes tipo "NDVI medio por término municipal" o "lámina de agua por embalse".


In [ ]:
from rasterstats import zonal_stats

# Nos quedamos con los municipios que intersectan el extent del raster, así no perdemos tiempo
from shapely.geometry import box
bbox = box(*src.bounds)
muni_aoi = municipios[municipios.intersects(bbox)].copy()
print(f'{len(muni_aoi)} municipios intersectan el extent del raster')


In [ ]:
stats = zonal_stats(
    muni_aoi,
    'ndvi_donana.tif',          # el raster derivado que guardamos antes
    stats=['mean', 'min', 'max', 'std', 'count'],
    nodata=np.nan,
)

import pandas as pd
muni_aoi = muni_aoi.reset_index(drop=True)
muni_aoi[['ndvi_mean', 'ndvi_min', 'ndvi_max', 'ndvi_std', 'n_pix']] = pd.DataFrame(stats)[
    ['mean', 'min', 'max', 'std', 'count']
]
muni_aoi.sort_values('ndvi_mean', ascending=False)[
    ['nombre', 'provincia', 'ndvi_mean', 'ndvi_std', 'n_pix']
].head(15)


In [ ]:
# Mapa coroplético: NDVI medio por municipio
fig, ax = plt.subplots(figsize=(11, 10))
muni_aoi.plot(column='ndvi_mean', ax=ax, cmap='RdYlGn', legend=True,
              legend_kwds={'label': 'NDVI medio', 'shrink': 0.6},
              edgecolor='gray', linewidth=0.3)
ax.set_title('NDVI medio por término municipal — área Doñana')
ax.set_axis_off()
plt.show()


> **Lectura crítica:** la fecha de la imagen condiciona enormemente el resultado. En invierno, los arrozales de Isla Mayor / La Puebla del Río salen con NDVI bajo (inundados o tierra desnuda) y en verano altísimo. Antes de comparar municipios, **siempre** ojea la fecha del satélite y el calendario de cultivos.


---
## 12. Ejercicio final integrador

Vas a calcular **el área inundada (en hectáreas) dentro del término municipal de Aznalcázar** (cubre buena parte de la marisma sur de Doñana). Pasos:

1. Filtra el GeoDataFrame `municipios` para quedarte con Aznalcázar.
2. Cruza el GeoDataFrame `gdf_agua` (polígonos de agua que ya calculamos) con Aznalcázar — usa `gpd.overlay(..., how='intersection')` o un `clip`.
3. Calcula el área total en hectáreas.
4. Pinta un mapa: el municipio en gris, los polígonos de agua dentro en azul.

> 🌟 **Bonus**: rehazlo aplicando antes la máscara `pixel_bueno` al MNDWI (`mndwi_limpio = np.where(pixel_bueno, mndwi, np.nan)`) y poligoniza esa nueva máscara. ¿Cuánta área "de agua" desaparece al filtrar nubes? Las nubes sobre marisma pueden dar falsos positivos en MNDWI porque las gotas tienen reflectancia parecida al agua en SWIR.


In [ ]:
# *** TU CÓDIGO AQUÍ ***
# 1) aznalcazar = ...
# 2) agua_en_aznalcazar = ...
# 3) area_ha = ...
# 4) plot


In [ ]:
# SOLUCIÓN — descomenta para verla
# aznalcazar = municipios[municipios['nombre'] == 'Aznalcázar']
# agua_en_aznalcazar = gpd.overlay(gdf_agua, aznalcazar, how='intersection')
# area_ha = agua_en_aznalcazar.area.sum() / 1e4
# print(f'Área inundada en Aznalcázar: {area_ha:.1f} ha')
#
# fig, ax = plt.subplots(figsize=(9, 9))
# aznalcazar.plot(ax=ax, color='lightgray', edgecolor='black')
# agua_en_aznalcazar.plot(ax=ax, color='steelblue', edgecolor='navy', linewidth=0.5)
# ax.set_title(f'Agua detectada en Aznalcázar: {area_ha:.0f} ha')
# ax.set_axis_off()
# plt.show()


---
## 13. Apéndice — Cómo preparamos la escena Landsat

> Este código **no se ejecuta en clase**, pero se incluye como referencia: enseña cómo se construyó `landsat_donana_marisma.tif` a partir de la escena completa L9 (banda por banda + QA_PIXEL) y el shapefile `rbios.shp` de la Reserva de la Biosfera de Doñana.

```python
import pathlib, numpy as np
import rasterio
from rasterio.mask import mask
import geopandas as gpd

ESC = pathlib.Path('20250327l9oli202_34')   # carpeta con la escena completa
RBIOS = pathlib.Path('rbios.shp')           # polígono de recorte

BANDAS_SR = [
    ('blue',  ESC / '20250327l9oli202_34_grn2_blue_b2.tif'),
    ('green', ESC / '20250327l9oli202_34_grn2_green_b3.tif'),
    ('red',   ESC / '20250327l9oli202_34_grn2_red_b4.tif'),
    ('nir',   ESC / '20250327l9oli202_34_grn2_nir_b5.tif'),
    ('swir1', ESC / '20250327l9oli202_34_grn2_swir1_b6.tif'),
    ('swir2', ESC / '20250327l9oli202_34_grn2_swir2_b7.tif'),
]
QA_PATH = ESC / '20250327l9oli202_34_fmask.tif'

rbios = gpd.read_file(RBIOS)
with rasterio.open(BANDAS_SR[0][1]) as ref:
    rbios = rbios.to_crs(ref.crs)
geom = [rbios.geometry.iloc[0].__geo_interface__]

# Recortar las 6 bandas SR
bandas_sr = []
for nombre, p in BANDAS_SR:
    with rasterio.open(p) as src:
        img, tr = mask(src, geom, crop=True, nodata=src.nodata, filled=True)
        arr = img[0].astype('float32')
        arr[arr == src.nodata] = np.nan
        bandas_sr.append(arr)
        ref_transform, ref_profile = tr, src.profile.copy()

# Recortar QA_PIXEL y unificar nodata (-9999 → NaN)
with rasterio.open(QA_PATH) as src:
    img, _ = mask(src, geom, crop=True, nodata=src.nodata, filled=True)
    qa_int = img[0].astype('int32')
qa = qa_int.astype('float32')
qa[qa_int == -9999] = np.nan

# Apilar y guardar
stack = np.stack(bandas_sr + [qa], axis=0)
H, W = stack.shape[1:]
profile = {
    'driver': 'GTiff', 'count': 7, 'dtype': 'float32',
    'height': H, 'width': W,
    'transform': ref_transform, 'crs': ref_profile['crs'],
    'nodata': np.nan,
    'compress': 'deflate', 'predictor': 3,
    'tiled': True, 'blockxsize': 256, 'blockysize': 256,
}
with rasterio.open('landsat_donana_marisma.tif', 'w', **profile) as dst:
    dst.write(stack)
    for i, n in enumerate(['blue (B2)','green (B3)','red (B4)','nir (B5)',
                           'swir1 (B6)','swir2 (B7)','qa_pixel'], 1):
        dst.set_band_description(i, n)
```

---

## Resumen del día

| Bloque | Función clave |
|--------|---------------|
| Abrir / metadatos | `rasterio.open(path)`, `.crs`, `.transform`, `.bounds`, `.profile` |
| Leer | `.read(n)`, `.read([1,2,3])`, `.read()` |
| Álgebra de bandas | aritmética NumPy elemento a elemento |
| Visualizar | `plt.imshow`, `rasterio.plot.show`, `np.dstack` para RGB |
| Guardar | `rasterio.open(..., 'w', **profile)` |
| Recortar por vectorial | `rasterio.mask.mask(src, geom, crop=True)` |
| Raster → vectorial | `rasterio.features.shapes(mask, transform=...)` |
| Estadísticas zonales | `rasterstats.zonal_stats(vector, raster, stats=[...])` |

---
*CHG — Curso de Python para el Análisis Espacial*
